# Player stat forecasts in the 2026 Six Nations

1. Which players are forecast to produce more of a stat than their positional peers?
2. For which stats does using player history improve forecast accuracy over a positional average?

We fit on all competitions before the opening match and hold the model fixed through the tournament. Each test observation is one recorded player appearance.

In [ ]:
from pathlib import Path

import pandas as pd

from six_nations_fantasy.data import load_dataset
from six_nations_fantasy.data.lineup import substitute_status
from six_nations_fantasy.models import PositionShrinkageMean
from six_nations_fantasy.models.position_shrinkage import STARTING_POSITIONS

data_dir = Path("../data/rebuild")  # relative to this notebook
data = load_dataset(data_dir)
players = pd.read_parquet(data_dir / "players.parquet")

In [ ]:
opening_day = "2026-02-05"
day_after_final = "2026-03-15"
matches = data.matches
training = data.select_matches(matches.loc[matches["date"] < opening_day, "match_id"])

tournament = matches["competition"].eq("Six Nations") & matches["date"].between(
    opening_day, day_after_final, inclusive="left"
)
test = data.select_matches(matches.loc[tournament, "match_id"]).player_matches()
fixtures = test[["match_id", "player_id", "number"]].assign(is_substitute=substitute_status(test))

## Forecast each stat

For each appearance, save the player forecast and the training average for the same starting position. All substitutes share a single bench baseline. The model uses the same shrinkage setting ($k=20$) for every stat. We keep the observed outcome alongside the forecasts for evaluation.

In [ ]:
stats = [
    "conversions", "penalty_goals", "drop_goals", "carries", "clean_breaks",
    "completed_tackles", "turnovers_conceded", "turnovers_won", "total_kicks",
    "passes", "carried_meters", "offloads", "defenders_beaten", "try_assists",
    "tries", "carries_per_minute", "total_tackles", "dominant_tackles",
    "ruck_turnovers", "lineouts_won", "tackles_per_minute",
]

position = pd.to_numeric(fixtures["number"]).map(STARTING_POSITIONS)
baseline_group = position.mask(fixtures["is_substitute"].fillna(False), "Substitute")

def forecast_stat(stat):
    model = PositionShrinkageMean(k=20, scorer=lambda rows: rows[stat])
    model.fit(training)
    return fixtures[["match_id", "player_id"]].assign(
        stat=stat,
        baseline_group=baseline_group,
        forecast=model.predict(fixtures),
        baseline=model.baselines(fixtures),
        actual=test[stat],
    )

forecasts = pd.concat([forecast_stat(stat) for stat in stats], ignore_index=True)

## Which players have high predicted volume relative to their peers?

A forecast-to-baseline ratio of 2 means twice the expected volume for that group. A large ratio can correspond to a small difference in events per match, especially for rare stats. The leaderboard shows both quantities. Higher volume can also be undesirable for stats such as turnovers conceded.

In [ ]:
stat_to_inspect = "try_assists"
leaderboard = (
    forecasts.loc[forecasts["stat"].eq(stat_to_inspect)]
    .groupby(["player_id", "baseline_group"])
    .agg(appearances=("match_id", "size"), forecast=("forecast", "mean"), baseline=("baseline", "mean"))
    .join(players["name"], on="player_id")
)
baseline = leaderboard["baseline"]
leaderboard["forecast_ratio"] = leaderboard["forecast"] / baseline.where(baseline > 0)
leaderboard["extra_per_match"] = leaderboard["forecast"] - baseline
leaderboard.nlargest(10, "forecast_ratio")

## For which stats does player history improve the forecast?

The **mean squared error skill score (MSESS)** measures the proportional reduction in mean squared error relative to the positional baseline. Compute the two errors across the same held-out appearances, then take their ratio.

$$\mathrm{MSESS} = 1 - \frac{\sum_i (y_i - \hat y_i)^2}{\sum_i (y_i - b_i)^2}$$

Here $y_i$ is the observed stat, $\hat y_i$ the player forecast and $b_i$ the baseline. A score of 0.10 means a 10% reduction in squared error. Zero means equal error and a negative value favors the baseline. A perfect baseline makes the score undefined. The RMSE columns give the typical scale of the errors in the original stat units.

In [ ]:
paired = forecasts.dropna(subset=["actual", "forecast", "baseline"]).copy()
paired["forecast_sq_error"] = (paired["actual"] - paired["forecast"]) ** 2
paired["baseline_sq_error"] = (paired["actual"] - paired["baseline"]) ** 2

accuracy = paired.groupby("stat").agg(
    appearances=("match_id", "size"),
    forecast_mse=("forecast_sq_error", "mean"),
    baseline_mse=("baseline_sq_error", "mean"),
).reindex(stats)
baseline_mse = accuracy["baseline_mse"]
accuracy["msess"] = 1 - accuracy["forecast_mse"] / baseline_mse.where(baseline_mse > 0)
accuracy["forecast_rmse"] = accuracy["forecast_mse"] ** 0.5
accuracy["baseline_rmse"] = baseline_mse ** 0.5
accuracy[["appearances", "forecast_rmse", "baseline_rmse", "msess"]].sort_values(
    "msess", ascending=False
)

Read the first table when comparing players for a particular stat. Read the second when deciding whether individual player history helps forecast that stat across this tournament. For fantasy selection, a further question is whether the players ranked highest before kickoff actually outperformed their positional peers.